In [9]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


file_path = './data/vote_data_8th.xlsx'

# 1. 0번째 줄을 머리글(Header)로 지정해서 불러오기
df = pd.read_excel(file_path, header=0)

# 2. 후보자 득표수는 필요 없으니, 맨 앞에서부터 6개 컬럼(시도 ~ 투표수)까지만 잘라내기
df = df.iloc[:, 0:6]

# 3. 컬럼명에 숨어있는 줄바꿈(\n)이나 띄어쓰기를 없애고 우리가 쓰기 편하게 이름 덮어쓰기
df.columns = ['시도', '구시군', '읍면동', '구분', '선거인수', '투표수']

# 4. 0번 인덱스 행 (후보자 이름이 적혀있던 쓰레기 줄) 영구 삭제
df = df.drop(0)

# 5. [핵심 기술] 세로 병합 때문에 생긴 빈칸(NaN)을 바로 위쪽에 있는 값으로 쭉 채워 내리기 (Forward Fill)
df[['시도', '구시군', '읍면동']] = df[['시도', '구시군', '읍면동']].ffill()

print("✨ 1차 세척이 완료된 깔끔한 데이터:")
display(df.head(10))

# 1. '구분' 칸이 비어있는(NaN) 행 삭제 (합계, 거소투표, 관외사전투표 등 제거)
# subset=['구분']은 "오직 '구분' 컬럼에 NaN이 있는지만 검사해서 지워라"라는 뜻입니다.
df_clean = df.dropna(subset=['구분']).copy()

# 2. '소계'도 파이썬으로 언제든 계산할 수 있으니 삭제!
# 우리는 오직 '관내사전투표'와 '선거일투표' 데이터만 남깁니다.
df_clean = df_clean[df_clean['구분'].isin(['관내사전투표', '선거일투표'])]

# 3. 중간중간 행을 지웠으니, 꼬여있는 인덱스 번호(0, 1, 2...)를 다시 0부터 예쁘게 재정렬합니다.
df_clean = df_clean.reset_index(drop=True)

print("✨ 2차 세척 완료! 불순물이 제거된 순도 100% 데이터:")
display(df_clean.head(10))


# # 1. 엑셀의 '피벗 테이블' 기능을 파이썬으로 구현
# df_pivot = df_clean.pivot_table(
#     index=['시도', '구시군', '읍면동'], 
#     columns='구분', 
#     values='투표수'
# )

# # 2. 피벗을 하면 컬럼명이 2층 구조로 바뀌는데, 이를 평평하게(1층으로) 펴줍니다.
# df_pivot = df_pivot.reset_index()

# # 3. 컬럼 이름을 우리가 보기 편하게 다시 정리합니다.
# df_pivot.columns = ['시도', '구시군', '읍면동', '사전투표수', '선거일투표수']

# print("✨ 최종 완성된 분석용 데이터(Wide Format):")
# display(df_pivot.head(10))


# 1. '투표수' 컬럼을 숫자로 강제 변환합니다. 
# 만약 숫자가 아닌 값이 섞여있다면 0으로 처리(coerce)합니다.
df_clean['투표수'] = pd.to_numeric(df_clean['투표수'], errors='coerce')

# (선택사항)혹시 NaN이 생겼다면 0으로 채워줍니다.
df_clean['투표수'] = df_clean['투표수'].fillna(0)

# 이제 피벗 테이블을 실행합니다.
df_pivot = df_clean.pivot_table(
    index=['시도', '구시군', '읍면동'], 
    columns='구분', 
    values='투표수',
    aggfunc='sum' # 덧셈을 하겠다는 명시적 표현
)

df_pivot = df_pivot.reset_index()
df_pivot.columns = ['시도', '구시군', '읍면동', '사전투표수', '선거일투표수']

print("✨ 최종 완성된 분석용 데이터(Wide Format):")
display(df_pivot.head(10))

# 1. output 폴더가 없으면 새로 만듭니다 (자동화의 기본!)
import os
if not os.path.exists('output'):
    os.makedirs('output')

# 2. 엑셀 파일로 저장 (index=False는 왼쪽의 0, 1, 2... 번호를 저장하지 않겠다는 뜻)
df_pivot.to_excel('./output/cleaned_vote_data.xlsx', index=False)

print("오늘의 미션 성공~! 데이터가 output 폴더에 저장되었습니다.")



✨ 1차 세척이 완료된 깔끔한 데이터:


,시도,구시군,읍면동,구분,선거인수,투표수
1,서울특별시,종로구,합계,NaN,"129,816","70,657"
2,서울특별시,종로구,거소투표,NaN,154,147
3,서울특별시,종로구,관외사전투표,NaN,"8,069","8,067"
4,서울특별시,종로구,청운효자동,소계,"9,179","4,887"
5,서울특별시,종로구,청운효자동,관내사전투표,"1,732","1,732"
6,서울특별시,종로구,청운효자동,선거일투표,"7,447","3,155"
7,서울특별시,종로구,사직동,소계,"8,057","4,452"
8,서울특별시,종로구,사직동,관내사전투표,"1,699","1,699"
9,서울특별시,종로구,사직동,선거일투표,"6,358","2,753"
10,서울특별시,종로구,삼청동,소계,"1,962","1,004"


✨ 2차 세척 완료! 불순물이 제거된 순도 100% 데이터:


,시도,구시군,읍면동,구분,선거인수,투표수
0,서울특별시,종로구,청운효자동,관내사전투표,"1,732","1,732"
1,서울특별시,종로구,청운효자동,선거일투표,"7,447","3,155"
2,서울특별시,종로구,사직동,관내사전투표,"1,699","1,699"
3,서울특별시,종로구,사직동,선거일투표,"6,358","2,753"
4,서울특별시,종로구,삼청동,관내사전투표,319,319
5,서울특별시,종로구,삼청동,선거일투표,"1,643",685
6,서울특별시,종로구,부암동,관내사전투표,"1,233","1,233"
7,서울특별시,종로구,부암동,선거일투표,"6,460","2,904"
8,서울특별시,종로구,평창동,관내사전투표,"2,449","2,449"
9,서울특별시,종로구,평창동,선거일투표,"12,281","5,482"


✨ 최종 완성된 분석용 데이터(Wide Format):


,시도,구시군,읍면동,사전투표수,선거일투표수
0,강원도,강릉시,강남동,0.0,0.0
1,강원도,강릉시,강동면,977.0,0.0
2,강원도,강릉시,경포동,651.0,0.0
3,강원도,강릉시,교1동,0.0,0.0
4,강원도,강릉시,교2동,0.0,0.0
5,강원도,강릉시,구정면,773.0,0.0
6,강원도,강릉시,내곡동,0.0,0.0
7,강원도,강릉시,사천면,0.0,0.0
8,강원도,강릉시,성덕동,0.0,0.0
9,강원도,강릉시,성산면,756.0,0.0


오늘의 미션 성공~! 데이터가 output 폴더에 저장되었습니다.
